# 05 - Extract aspects from comments

Finds which laptop aspects (battery, screen, RAM, ...) each comment mentions, using the regex patterns in `src/opinion_owl/aspect_patterns.py`. Notebook 06 then predicts the sentiment of each (comment, aspect) pair with all three models.
**Input:** `data/comments/comments_test.csv` (built-in test comments with expected labels) or `data/comments/comments_raw.csv` (a real CSV from the browser extension; needs `comment_id` and `text` columns)
**Output:** `data/comments/comments_aspects.csv`, one row per (comment, aspect)

Set `USE_TEST_DATA = True` to check that the patterns find what they should. Set it to `False` to run on real comments.

In [1]:
import pandas as pd

from opinion_owl.aspect_patterns import ASPECT_PATTERNS, COMPILED
from opinion_owl.config import COMMENTS_DIR

USE_TEST_DATA = True

INPUT_PATH = COMMENTS_DIR / ("comments_test.csv" if USE_TEST_DATA else "comments_raw.csv")
OUTPUT_PATH = COMMENTS_DIR / "comments_aspects.csv"

print(len(ASPECT_PATTERNS), "aspects,", sum(len(p) for p in ASPECT_PATTERNS.values()), "patterns")

43 aspects, 411 patterns


## 1. Test comments

In [2]:
# (comment_id, text, expected aspects separated by "|")
# Written to sound like real Reddit comments. Each one lists the aspects the patterns
# should find, so the check in step 4 can count misses and false alarms.
TEST_COMMENTS = [
    ('t01', "Bought this for college last year. 8GB RAM is honestly not enough once you open 20 chrome tabs, but it's fine for notes and Word.", 'ram|school|productivity|study'),
    ('t02', "Don't expect AAA gaming on this thing, the iGPU just can't keep up. Minecraft and Roblox run fine though.", 'gaming performance|graphics performance|light gaming'),
    ('t03', 'Battery lasts like 9 hours of web browsing, and the USB-C charging is a lifesaver when I travel.', 'battery|productivity|ports|travel use'),
    ('t04', "The screen is dim, only 250 nits. Outdoors it's basically unusable. 1080p IPS at least.", 'screen'),
    ('t05', 'Hinge feels wobbly after 6 months and the plastic body creaks when I pick it up.', 'hinge quality|build quality'),
    ('t06', 'Got it for $450 on sale, honestly great value for money for a student.', 'value for money|school'),
    ('t07', 'Fans get so loud when I compile code in WSL2. Thermals are bad, hits 95C and throttles.', 'noise|thermals|software development'),
    ('t08', 'Keyboard is mushy and the trackpad has no palm rejection. Typing on it all day sucks.', 'keyboard|trackpad'),
    ('t09', 'Is the RAM soldered? I want to upgrade to 32GB later. Two M.2 slots would be nice too.', 'ram|upgradeability|storage'),
    ('t10', 'Perfect for office work: Excel, Teams meetings, and a ton of PDFs. Never lags.', 'office work|productivity|general performance'),
    ('t11', 'The touchscreen with the stylus is great for note-taking in lectures, 2-in-1 is a game changer.', 'touchscreen|study'),
    ('t12', 'Runs Ollama with a 7B model okay-ish, but the NPU is useless for now. Copilot+ is marketing.', 'ai performance'),
    ('t13', '512GB SSD fills up fast if you install a few Steam games. Wish it came with 1TB.', 'storage'),
    ('t14', 'Ryzen 5 7530U is plenty quick for everyday tasks. Core i5 versions cost more.', 'cpu performance|general performance|value for money'),
    ('t15', 'Speakers are tinny and the webcam is a grainy 720p. Video calls look awful.', 'speakers|webcam'),
    ('t16', 'Only two USB-A ports and an HDMI. No ethernet, no SD card reader, need a dongle.', 'ports'),
    ('t17', 'Weighs 1.3 kg, super portable. Slim enough to slide in my backpack.', 'portability'),
    ('t18', 'Wi-Fi keeps disconnecting and Bluetooth is flaky with my headphones.', 'connectivity'),
    ('t19', 'Windows 11 comes with so much bloatware, I did a clean install and the drivers worked fine.', 'operating system|bloatware'),
    ('t20', "Linux (Ubuntu) works well, but the fingerprint reader doesn't.", 'operating system|security'),
    ('t21', 'Warranty service was a nightmare. RMA took 5 weeks and they sent it back with a faulty screen.', 'customer support|screen'),
    ('t22', 'Overpriced for what you get. A refurbished ThinkPad is a way better deal.', 'value for money|brand|refurbished / used'),
    ('t23', 'Plays Valorant at 144 FPS on low settings, but Cyberpunk at 30 FPS is a slideshow.', 'gaming performance'),
    ('t24', '120Hz OLED panel looks stunning, colour accuracy is great too.', 'refresh rate|screen'),
    ('t25', 'Video editing in Premiere is painful with this, the render times are huge.', 'content creation'),
    ('t26', "Got a 14 inch version, it's the perfect size and the 16:10 aspect ratio helps.", 'screen size'),
    ('t27', 'Looks sleek and the bezels are tiny, but it feels cheap and flimsy.', 'design|build quality'),
    ('t28', 'The laptop crashes randomly, got a blue screen twice this week. Not reliable.', 'reliability'),
    ('t29', 'I use it for data science and Jupyter notebooks, Docker runs fine with 16GB.', 'software development|ram'),
    ('t30', 'Ran my old games from 2012 and some indie games with zero issues. Not a gaming laptop though.', 'light gaming|gaming performance'),
    ('t31', 'Runs hot even when browsing. Cooling is a joke and the fan never stops.', 'thermals|noise|productivity'),
    ('t32', 'Just ok. Nothing special.', ''),
    ('t33', 'Anyone know if Rosetta handles x86 apps well on the M2 MacBook?', 'software compatibility|cpu performance|brand'),
    ('t34', 'Great for a cs major: VS Code, Python, and cloud gaming on GeForce Now after class.', 'software development|school|light gaming'),
    ('t35', 'Rs 55,000 is a steal for 16GB RAM and 512GB SSD with a Ryzen 7.', 'value for money|ram|storage|cpu performance'),
    ('t36', 'Charger is bulky but fast charging gets me to 50% in 30 minutes.', 'battery'),
    ('t37', 'Sturdy aluminium chassis and great aesthetics, build quality beats the Dell.', 'build quality|design|brand'),
    ('t38', "I'd skip this one, the display flickers and has dead pixels out of the box.", 'screen|reliability'),
    ('t39', 'Perfect budget laptop for homework, YouTube and Netflix.', 'value for money|school|productivity'),
    ('t40', "The assignment deadline is tomorrow and I still can't get Wi-Fi.", 'study|connectivity'),
]

if USE_TEST_DATA:
    pd.DataFrame(TEST_COMMENTS, columns=["comment_id", "text", "expected_aspects"]).to_csv(INPUT_PATH, index=False)

## 2. Load the comments

In [3]:
comments = pd.read_csv(INPUT_PATH)
comments["text"] = comments["text"].fillna("").astype(str).str.strip()
comments = comments[comments["text"] != ""].drop_duplicates("comment_id").reset_index(drop=True)
print("Comments:", len(comments))

Comments: 40


## 3. Find the aspects

In [4]:
def extract_aspects(comment_id, text):
    """Return one row per distinct (aspect, matched words) found in the text.

    raw_aspect keeps the exact words that matched ("8gb", "usb-c"), which makes it
    easy to see why a pattern fired when debugging.
    """
    rows, seen = [], set()
    for aspect, regexes in COMPILED.items():
        for regex in regexes:
            for match in regex.finditer(text):
                raw = match.group(0).lower().strip()
                if (aspect, raw) in seen:
                    continue
                seen.add((aspect, raw))
                rows.append({"comment_id": comment_id, "text": text, "raw_aspect": raw, "aspect_term": aspect})
    return rows


aspect_rows = [
    row
    for comment_id, text in zip(comments["comment_id"], comments["text"])
    for row in extract_aspects(comment_id, text)
]
aspects_df = pd.DataFrame(aspect_rows, columns=["comment_id", "text", "raw_aspect", "aspect_term"])
print("Aspect rows:", len(aspects_df))
aspects_df.head(10)

Aspect rows: 167


,comment_id,text,raw_aspect,aspect_term
0,t01,Bought this for college last year. 8GB RAM is ...,ram,ram
1,t01,Bought this for college last year. 8GB RAM is ...,8gb,ram
2,t01,Bought this for college last year. 8GB RAM is ...,college,school
3,t01,Bought this for college last year. 8GB RAM is ...,notes,study
4,t01,Bought this for college last year. 8GB RAM is ...,chrome,productivity
5,t01,Bought this for college last year. 8GB RAM is ...,tabs,productivity
6,t02,"Don't expect AAA gaming on this thing, the iGP...",igpu,graphics performance
7,t02,"Don't expect AAA gaming on this thing, the iGP...",aaa,gaming performance
8,t02,"Don't expect AAA gaming on this thing, the iGP...",roblox,light gaming
9,t02,"Don't expect AAA gaming on this thing, the iGP...",minecraft,light gaming


## 4. Check against the expected labels (test data only)

In [5]:
# TP = aspects found and expected, FP = found but not expected, FN = expected but missed.
# Mismatches are printed with the comment so you can see which pattern to fix.
if "expected_aspects" in comments.columns:
    detected = aspects_df.groupby("comment_id")["aspect_term"].agg(set).to_dict()
    tp = fp = fn = 0

    for _, row in comments.iterrows():
        expected = set(filter(None, str(row["expected_aspects"]).replace("nan", "").split("|")))
        found = detected.get(row["comment_id"], set())
        tp += len(expected & found)
        fp += len(found - expected)
        fn += len(expected - found)

        if expected != found:
            print(row["comment_id"], "| MISSED:", sorted(expected - found), "| EXTRA:", sorted(found - expected))
            print("   ", row["text"])

    print(f"\nTP={tp} FP={fp} FN={fn}  precision={tp / max(tp + fp, 1):.2f}  recall={tp / max(tp + fn, 1):.2f}")


TP=84 FP=0 FN=0  precision=1.00  recall=1.00


## 5. Which aspects fire most

In [6]:
# On real comments, aspects with far too many mentions usually have a pattern that is too broad.
mention_counts = (
    aspects_df.groupby("aspect_term").size().sort_values(ascending=False).rename("mentions").reset_index()
)
mention_counts

,aspect_term,mentions
0,screen,9
1,build quality,8
2,productivity,8
3,software development,8
4,ports,8
5,value for money,8
6,ram,7
7,light gaming,6
8,gaming performance,6
9,storage,6


## 6. Save one row per (comment, aspect)

In [7]:
# Several patterns can match in one comment for the same aspect (for example "chrome" and
# "tabs" both mean productivity), so they are merged into one row with the words joined.
model_input = (
    aspects_df.groupby(["comment_id", "aspect_term", "text"])["raw_aspect"]
    .agg(lambda words: ", ".join(sorted(set(words))))
    .reset_index()
)
# Kept for older notebooks; notebook 06 builds its own model input.
model_input["tfidf_input"] = model_input["aspect_term"] + " [SEP] " + model_input["text"]

model_input.to_csv(OUTPUT_PATH, index=False)
print("Saved", len(model_input), "rows to", OUTPUT_PATH)
model_input.head(10)

Saved 84 rows to C:\Programming\opinion-owl\data\comments\comments_aspects.csv


,comment_id,aspect_term,text,raw_aspect,tfidf_input
0,t01,productivity,Bought this for college last year. 8GB RAM is ...,"chrome, tabs",productivity [SEP] Bought this for college las...
1,t01,ram,Bought this for college last year. 8GB RAM is ...,"8gb, ram",ram [SEP] Bought this for college last year. 8...
2,t01,school,Bought this for college last year. 8GB RAM is ...,college,school [SEP] Bought this for college last year...
3,t01,study,Bought this for college last year. 8GB RAM is ...,notes,study [SEP] Bought this for college last year....
4,t02,gaming performance,"Don't expect AAA gaming on this thing, the iGP...",aaa,gaming performance [SEP] Don't expect AAA gami...
5,t02,graphics performance,"Don't expect AAA gaming on this thing, the iGP...",igpu,graphics performance [SEP] Don't expect AAA ga...
6,t02,light gaming,"Don't expect AAA gaming on this thing, the iGP...","minecraft, roblox",light gaming [SEP] Don't expect AAA gaming on ...
7,t03,battery,"Battery lasts like 9 hours of web browsing, an...","battery, charging",battery [SEP] Battery lasts like 9 hours of we...
8,t03,ports,"Battery lasts like 9 hours of web browsing, an...",usb-c,ports [SEP] Battery lasts like 9 hours of web ...
9,t03,productivity,"Battery lasts like 9 hours of web browsing, an...","browsing, web browsing",productivity [SEP] Battery lasts like 9 hours ...
